# ⚽ Pipeline de Generación de Datos de Eventos de Fútbol

**Football Tracking → Event Data — Google Colab**

Este notebook ejecuta el pipeline completo de principio a fin:

1. **Configuración** — clonar el repositorio e instalar dependencias.
2. **Google Drive** — montar Drive para subir el vídeo y guardar resultados.
3. **Subir vídeo** — desde tu ordenador o desde una ruta de Drive.
4. **Tracking** — detección de jugadores y balón con YOLO/Roboflow (`data_cleanup/main.py`).
5. **Generación de eventos** — pases, tiros, goles, balones perdidos, saques, etc.
6. **Visualización** — tabla resumen y mapa de eventos sobre el campo.
7. **Descargar resultados** — CSV de tracking y CSV de eventos.

> 📌 **Proyecto universitario de analítica de fútbol.** Cada sección incluye una breve explicación en español.

> ⚠️ **Importante:** activa la GPU antes de empezar en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*.

## 0. Comprobación de GPU

El tracking con YOLO es **mucho** más rápido con GPU. Esta celda avisa si no estás usando una.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"\u2705 GPU activa: {torch.cuda.get_device_name(0)}")
else:
    print("\u26a0\ufe0f  No se ha detectado GPU. El tracking ser\u00e1 MUY lento.")
    print("   Ve a: Entorno de ejecuci\u00f3n \u2192 Cambiar tipo de entorno \u2192 Acelerador por hardware \u2192 GPU")

## 1. Configuración: clonar repositorio e instalar dependencias

Clonamos el repositorio e instalamos `requirements.txt` junto con `filterpy` y `scipy`.

> 🔑 **Usa tu propio fork.** El código de generación de eventos y `data_cleanup/main.py` viven en **tu** fork del proyecto, no en el repositorio original. Sube esta rama a tu fork de GitHub y pon su URL en `REPO_URL`. La celda comprueba al final que el código de eventos está presente.

In [ ]:
import os

# \u2b07\ufe0f  Cambia esto por la URL de TU fork (debe contener el c\u00f3digo de event_generation).
REPO_URL = "https://github.com/TU_USUARIO/FootballTrackingDataGeneration.git"
BRANCH = "main"
REPO_DIR = "/content/FootballTrackingDataGeneration"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Dependencias del proyecto + las del generador de eventos.
!pip install -q -r requirements.txt
!pip install -q filterpy scipy

# Verificaci\u00f3n: el repo clonado DEBE incluir el generador de eventos.
assert os.path.exists("data_cleanup/lib/event_generator.py"), (
    "\u274c El repositorio clonado no contiene 'data_cleanup/lib/event_generator.py'.\n"
    "   Sube esta rama a tu fork y ajusta REPO_URL / BRANCH.")
assert os.path.exists("data_cleanup/main.py"), (
    "\u274c Falta 'data_cleanup/main.py' en el repo clonado. Usa tu fork actualizado.")
print("\u2705 Repositorio y dependencias listos.")

## 2. Montar Google Drive

Montamos Drive para poder **subir vídeos** y **guardar los resultados** de forma persistente (si no, se pierden al cerrar Colab). Todo se guardará en la carpeta `football_analytics` de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/football_analytics"
TRACKING_DIR = os.path.join(DRIVE_DIR, "tracking_output")
EVENTS_DIR = os.path.join(DRIVE_DIR, "event_output")
for d in (DRIVE_DIR, TRACKING_DIR, EVENTS_DIR):
    os.makedirs(d, exist_ok=True)
print("\U0001f4c1 Resultados se guardar\u00e1n en:", DRIVE_DIR)

## 2b. Modelos de detección (YOLOv8)

El tracking usa **YOLOv8 (ultralytics)**. Los pesos `yolov8n.pt` se descargan automáticamente la primera vez, así que **no necesitas ninguna clave de API**.

> 💡 `yolov8n.pt` es un modelo genérico (COCO): detecta personas ("jugadores") y el "balón", pero no distingue porteros/árbitros ni proyecta a coordenadas reales del campo. Para mejores resultados, entrena un modelo de fútbol y pásalo con `--player-model tus_pesos.pt`.

In [ ]:
# YOLOv8 no requiere clave de API: los pesos se descargan solos al ejecutar el tracking.
print("✅ Usando YOLOv8 (ultralytics). No se necesita clave de API.")

## 3. Subir el vídeo del partido

Tienes dos opciones:
- **Opción A — Drive:** pon la ruta del vídeo en `DRIVE_VIDEO_PATH` (p. ej. `/content/drive/MyDrive/mi_partido.mp4`).
- **Opción B — Subida directa:** deja `DRIVE_VIDEO_PATH` vacío y se abrirá un selector de archivos.

> 💡 Para clips cortos (30–60 s) el tracking es bastante más rápido. Formato esperado: `.mp4`.

In [ ]:
# Opci\u00f3n A: ruta a un v\u00eddeo ya en tu Drive (deja vac\u00edo para subir desde el ordenador).
DRIVE_VIDEO_PATH = ""

if DRIVE_VIDEO_PATH:
    assert os.path.exists(DRIVE_VIDEO_PATH), f"No existe: {DRIVE_VIDEO_PATH}"
    VIDEO_PATH = DRIVE_VIDEO_PATH
else:
    from google.colab import files
    print("Selecciona un archivo .mp4 desde tu ordenador...")
    uploaded = files.upload()
    VIDEO_PATH = os.path.abspath(list(uploaded.keys())[0])

VIDEO_NAME = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
print("\U0001f3ac V\u00eddeo seleccionado:", VIDEO_PATH)

## 4. Ejecutar el tracking (`data_cleanup/main.py`)

Ejecuta la detección de jugadores y balón con YOLO sobre cada fotograma y proyecta las posiciones al campo en 2D. El resultado es un **CSV de tracking** que se guarda en tu Drive.

**Mejoras de detección del balón** (resuelven el problema de pocos eventos por balón perdido):
- `--ball-model football`: descarga un modelo YOLOv8 entrenado con fútbol de TV (clases `ball/goalkeeper/player/referee`) desde Hugging Face, sin API key. Si falla la descarga, usa `yolov8m.pt` (mediano, mucho mejor que nano para objetos pequeños).
- `--ball-conf 0.15`: umbral de confianza bajo para no descartar el balón borroso o lejano.
- `--ball-interp-gap 15`: interpola linealmente la posición del balón en huecos de hasta 15 fotogramas, para que unas pocas detecciones perdidas no reinicien la posesión.

> ⏱️ Este paso es el más lento (depende de la duración del vídeo y de la GPU).

In [ ]:
cmd = (
    f'python data_cleanup/main.py '
    f'--video "{VIDEO_PATH}" '
    f'--output "{TRACKING_DIR}" '
    # Modelo de balón específico de fútbol (broadcast, vía HF; fallback yolov8m.pt).
    f'--ball-model football '
    # Confianza baja: el balón es pequeño/rápido y se pierde con conf alta.
    f'--ball-conf 0.15 '
    # Interpola el balón en huecos de hasta 15 fotogramas (evita reiniciar la posesión).
    f'--ball-interp-gap 15'
)
print(cmd)
!{cmd}

TRACKING_CSV = os.path.join(TRACKING_DIR, VIDEO_NAME + ".csv")
assert os.path.exists(TRACKING_CSV), "No se generó el CSV de tracking. Revisa la salida de arriba."
print("\u2705 Tracking CSV:", TRACKING_CSV)

## 5. Generar los datos de eventos

Cargamos el tracking con `Match.import_raw_data` y ejecutamos el generador de eventos (`match.generate_events()`). Detecta posesiones y las clasifica en eventos (PASE, TIRO, GOL, BALÓN PERDIDO, RECUPERACIÓN, SAQUE, etc.) y exporta un **CSV en formato de eventos de Metrica**.

> 🔧 Puedes ajustar el radio de posesión (en metros) con `match.generate_events(possession_radius=2.0)` según lo limpio que sea tu tracking.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, "data_cleanup"))
from lib.match import Match

match = Match()
match.import_raw_data(os.path.dirname(TRACKING_CSV) + os.sep, os.path.basename(TRACKING_CSV))
print(f"Importados {match.frames} fotogramas y {len(match.players)} objetos rastreados.")

events = match.generate_events()
print("\nResumen de eventos:", events.summary())

EVENTS_CSV = os.path.join(EVENTS_DIR, VIDEO_NAME + "_events.csv")
events.export(path=EVENTS_DIR + os.sep, file_name=os.path.basename(EVENTS_CSV))
print("\u2705 Eventos exportados a:", EVENTS_CSV)

## 6. Visualización

Mostramos (a) una **tabla resumen** con el recuento por tipo de evento y la lista completa, y (b) un **mapa del campo** con la posición de cada evento coloreada por tipo.

In [ ]:
import pandas as pd

df = pd.read_csv(EVENTS_CSV)

print("Recuento por tipo de evento:")
display(df["Type"].value_counts().rename_axis("Evento").to_frame("Cantidad"))

print("\nListado completo de eventos:")
display(df)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

def draw_pitch(ax):
    """Dibuja un campo simple en coordenadas normalizadas [0, 1] x [0, 1]."""
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, color="black", lw=2))
    ax.plot([0.5, 0.5], [0, 1], color="black", lw=1)              # línea de medio campo
    ax.add_patch(Circle((0.5, 0.5), 0.083, fill=False, color="black", lw=1))  # círculo central
    # Áreas
    ax.add_patch(plt.Rectangle((0, 0.21), 0.16, 0.58, fill=False, color="black", lw=1))
    ax.add_patch(plt.Rectangle((0.84, 0.21), 0.16, 0.58, fill=False, color="black", lw=1))
    # Porterías
    ax.add_patch(plt.Rectangle((-0.01, 0.45), 0.01, 0.10, color="black"))
    ax.add_patch(plt.Rectangle((1.0, 0.45), 0.01, 0.10, color="black"))
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(1.08, -0.08)  # y invertida: igual que los datos de tracking
    ax.set_aspect(68.0 / 105.0)
    ax.axis("off")

plot_df = df.dropna(subset=["Start X", "Start Y"]).copy()
event_types = sorted(plot_df["Type"].unique())
palette = list(plt.cm.tab10.colors)  # 10 colores fijos, compatible con todas las versiones
colors = {t: palette[i % len(palette)] for i, t in enumerate(event_types)}

fig, ax = plt.subplots(figsize=(12, 8))
ax.add_patch(plt.Rectangle((0, 0), 1, 1, color="#3a8a3a", alpha=0.12, zorder=0))
draw_pitch(ax)

for t in event_types:
    sub = plot_df[plot_df["Type"] == t]
    ax.scatter(sub["Start X"], sub["Start Y"], s=120, color=colors[t],
               edgecolors="black", linewidths=0.6, label=f"{t} ({len(sub)})", zorder=3)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False)
ax.set_title(f"Eventos detectados — {VIDEO_NAME}", fontsize=14)
plt.tight_layout()
fig_path = os.path.join(EVENTS_DIR, VIDEO_NAME + "_event_map.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("🗺️ Mapa guardado en:", fig_path)

## 7. Descargar resultados

Los CSV ya están en tu Drive (carpeta `football_analytics`). Esta celda también te ofrece la **descarga directa** del CSV de tracking y del CSV de eventos.

In [ ]:
from google.colab import files

print("\U0001f4be Tambi\u00e9n guardados en Drive:", DRIVE_DIR)
print("   -", TRACKING_CSV)
print("   -", EVENTS_CSV)

files.download(TRACKING_CSV)
files.download(EVENTS_CSV)